# Reviewer checks: leakage, design and inference checks (real data only)
Run All (about 20-40 minutes). Version 2: fixes the statsmodels/SciPy import error, adds a fixed-model calibration-age check (R3), a Lending Club charge-off lag sweep (R10b) and real-time monitoring gap (R13, both need `extra_data/label_dates.csv`), and Lending Club rows for R14/R15 (need `extra_data/payments.csv`). Sections are isolated; the last cell zips everything to `Desktop/Q2/outputs_reviewer.zip`.

| id | check |
|---|---|
| R1 | rolling-origin rows use a model trained on years **before** the calibration year (no leakage) |
| R2 | coverage error versus the **change** in calibration gap (test gap minus calibration-year gap) versus the test gap alone |
| R3 | calibration age versus calendar drift: test-month fixed effects with several calibration years |
| R4 | permutation test: is each deviation distinguishable from exchangeability? |
| R5-R7 | month-block bootstrap intervals (with calibration-set resampling) for the Prosper validity table and for all segments; Benjamini-Hochberg correction; HAC / block-bootstrap slopes |
| R8 | is the validation-year model choice distinguishable from noise? |
| R9, R11 | delayed-feedback gain sweep; all loans and several resamples instead of one 2,500-per-month sample |
| R10 | label-lag sweep (Prosper, exact dates, extra charge-off lag of 0-8 months added) |
| R12 | Lending Club with a 2,047-loan calibration set (same size as Prosper design B) |
| R13 | monitoring alarm: threshold sweep, precision/recall, real-time (known-label) gap on Prosper |
| R14 | reviewer break-even with separate accuracy on repayers and defaulters |
| R15 | Spearman vs AUC: why a positive rank correlation can coexist with AUC below 0.5 |
| R16 | share of empty prediction sets |
| R17 | recomputation of in-text numbers from the data |

In [ ]:
import os, sys, subprocess, importlib, warnings, json, hashlib, time, traceback, contextlib, shutil, urllib.request
from pathlib import Path
def ensure(pkg, mod=None):
    try: importlib.import_module(mod or pkg)
    except ImportError: subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])
ensure('lightgbm'); ensure('statsmodels'); ensure('scikit-learn', 'sklearn')
# statsmodels older than 0.14.5 cannot be imported with SciPy 1.16+ ("cannot import name '_lazywhere'"): upgrade it automatically
try:
    import statsmodels.api
except ImportError as _e:
    print('statsmodels does not work with this SciPy (', _e, ') -> upgrading statsmodels ...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'statsmodels>=0.14.5'])
    for _k in [k for k in sys.modules if k.startswith('statsmodels')]: del sys.modules[_k]
    importlib.invalidate_caches()
    try: import statsmodels.api
    except ImportError: raise SystemExit('statsmodels was upgraded: please use Kernel > Restart Kernel and Run All Cells')
import numpy as np, pandas as pd, lightgbm as lgb, matplotlib
matplotlib.use(os.environ.get('MPL_BACKEND', matplotlib.get_backend()))
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, log_loss
from sklearn.linear_model import LogisticRegression
from statsmodels.stats.proportion import proportion_confint
warnings.filterwarnings('ignore'); plt.style.use('seaborn-v0_8-whitegrid')
def find_q2():
    if os.environ.get('Q2_DIR'): return Path(os.environ['Q2_DIR'])
    home = Path.home()
    for d in [home/'Desktop', home/'OneDrive'/'Desktop', *sorted(home.glob('OneDrive*/Desktop'))]:
        if (d/'Q2').is_dir(): return d/'Q2'
Q2 = find_q2(); assert Q2 is not None, "No 'Q2' folder on the Desktop; set Q2 = Path(r'...') here."
OUT = Q2/'outputs_reviewer'; TAB, FIG = OUT/'tables', OUT/'figures'; TAB.mkdir(parents=True, exist_ok=True); FIG.mkdir(exist_ok=True)
SEED, ALPHA = 42, 0.10; FAILED = []; T0 = time.time()
@contextlib.contextmanager
def section(name):
    t = time.time(); print(f"\n{'=' * 10} {name} {'=' * 10}")
    try: yield
    except Exception:
        FAILED.append(name); print(f"!!! SECTION FAILED: {name}"); traceback.print_exc()
    finally: print(f"[{name}: {time.time() - t:.0f}s | total {(time.time() - T0) / 60:.1f} min]")
def save(df, name): df.round(5).to_csv(TAB/f'{name}.csv', index=False)
def savefig(name):
    for ext in ('png', 'pdf'): plt.savefig(FIG/f'{name}.{ext}', dpi=180, bbox_inches='tight')
def conf_q(scores, alpha):
    s = np.sort(np.asarray(scores)); n = len(s); k = int(np.ceil((n + 1) * (1 - alpha)))
    return np.inf if k > n else s[k - 1]
def true_scores(p, y): return np.where(y == 1, 1 - p, p)
def conformal_sets(p, y, ca, alpha=ALPHA, nbin=5):
    sc = true_scores(p, y); q = conf_q(sc[ca], alpha); q0, q1 = conf_q(sc[ca & (y == 0)], alpha), conf_q(sc[ca & (y == 1)], alpha)
    edges = np.quantile(p[ca], np.linspace(0, 1, nbin + 1)[1:-1]); b = np.digitize(p, edges)
    qb = np.array([conf_q(sc[ca & (b == k)], alpha) for k in range(nbin)])[b]
    return {'static': np.column_stack([p <= q, (1 - p) <= q]), 'class-conditional': np.column_stack([p <= q0, (1 - p) <= q1]), 'bin-conditional': np.column_stack([p <= qb, (1 - p) <= qb])}
def cov_stats(S, y):
    hit = S[np.arange(len(y)), y]; ap, rj = S[:, 0] & ~S[:, 1], S[:, 1] & ~S[:, 0]
    return dict(cov=hit.mean(), cov_default=hit[y == 1].mean() if (y == 1).any() else np.nan, size=S.sum(1).mean(), share_review=1 - ap.mean() - rj.mean(), share_reject_only=rj.mean())
def tiers(S): ap = S[:, 0] & ~S[:, 1]; rj = S[:, 1] & ~S[:, 0]; return ap, ~(ap | rj), rj
def wilson(k, n): return proportion_confint(int(k), int(n), method='wilson')
print("Q2:", Q2)
from statsmodels.stats.multitest import multipletests
import statsmodels.api as sm

In [ ]:
# ---------- 0. extraction: Prosper file -> analysis frame ----------
with section('0. data'):
    PF = Q2/'extra_data'/'prosper'/'prosperLoanData.csv'; PF.parent.mkdir(parents=True, exist_ok=True)
    if not PF.exists():
        url = "https://raw.githubusercontent.com/chukwume/prosperloandata/master/prosperLoanData.csv"; print("downloading", url)
        urllib.request.urlretrieve(url, PF)
    sha = hashlib.sha256(PF.read_bytes()).hexdigest(); print("SHA-256:", sha, "| matches expected:", sha == 'fce13408ac22e8201363e71e1a16d9795305f4aaf8c0f67efcd80424bc5b9e02')
    d = pd.read_csv(PF, low_memory=False); print("raw rows:", len(d))
    dt = lambda c: pd.to_datetime(d[c], format='ISO8601', errors='coerce')
    d['orig'], d['closed'], d['listed'], d['first_cl'] = dt('LoanOriginationDate'), dt('ClosedDate'), dt('ListingCreationDate'), dt('FirstRecordedCreditLine')
    keep = d['LoanStatus'].isin(['Completed', 'Chargedoff', 'Defaulted']) & (d['Term'] == 36) & (d['orig'] >= '2006-01-01') & (d['orig'] <= '2011-03-31')
    f = d[keep].sort_values('orig', kind='stable').reset_index(drop=True); print(f"resolved 36-month loans, issued 2006 to Mar-2011: {len(f):,}  default rate {f['LoanStatus'].ne('Completed').mean():.2%}")
    f['default'] = f['LoanStatus'].ne('Completed').astype(int); f['year'], f['month'] = f['orig'].dt.year, f['orig'].dt.month
    f['tm'] = (f['year'] - 2006) * 12 + f['month']; f['reveal'] = np.maximum(((f['closed'].dt.year - 2006) * 12 + f['closed'].dt.month).values, f['tm'].values + 1)
    f['funded'] = f['LoanOriginalAmount'].astype(float)
    f['profit'] = f['LP_InterestandFees'] + f['LP_ServiceFees'] + f['LP_CollectionFees'] + f['LP_NonPrincipalRecoverypayments'] - f['LP_NetPrincipalLoss']
    f['roi'] = f['profit'] / f['funded']
    gmap = {'AA': 0, 'A': 1, 'B': 2, 'C': 3, 'D': 4, 'E': 5, 'HR': 6}
    f['grade_ord'] = f['ProsperRating (Alpha)'].map(gmap).fillna(f['CreditGrade'].map(gmap))
    f['fico'] = (f['CreditScoreRangeLower'] + f['CreditScoreRangeUpper']) / 2; f['dti'] = f['DebtToIncomeRatio'].clip(0, 1.5)
    f['log_income'] = np.log1p(f['StatedMonthlyIncome'].clip(0, 100000)); f['log_amount'] = np.log1p(f['funded']); f['rate'] = f['BorrowerRate']
    f['homeowner'] = f['IsBorrowerHomeowner'].astype(float); f['employment'] = f['EmploymentStatus'].fillna('NA'); f['purpose'] = f['ListingCategory (numeric)'].astype(str); f['state'] = f['BorrowerState'].fillna('NA')
    f['bankcard_util'] = f['BankcardUtilization'].clip(0, 2); f['log_revbal'] = np.log1p(f['RevolvingCreditBalance'].clip(lower=0)); f['log_amt_delinq'] = np.log1p(f['AmountDelinquent'].clip(lower=0)); f['log_avail_bc'] = np.log1p(f['AvailableBankcardCredit'].clip(lower=0))
    f['credit_hist_yrs'] = (f['listed'] - f['first_cl']).dt.days / 365.25
    for a, b in (('open_lines', 'OpenCreditLines'), ('curr_delinq', 'CurrentDelinquencies'), ('delinq_7y', 'DelinquenciesLast7Years'), ('pub_rec_10y', 'PublicRecordsLast10Years'), ('inq_6m', 'InquiriesLast6Months'),
                 ('total_inq', 'TotalInquiries'), ('total_trades', 'TotalTrades'), ('never_delinq_pct', 'TradesNeverDelinquent (percentage)'), ('curr_lines', 'CurrentCreditLines')): f[a] = f[b]
    f['prior_prosper_loans'] = f['TotalProsperLoans'].fillna(0)
    CORE = ['fico', 'dti', 'log_income', 'log_amount', 'homeowner', 'inq_6m', 'bankcard_util', 'curr_delinq', 'open_lines', 'log_revbal']
    BUREAU = ['log_amt_delinq', 'delinq_7y', 'pub_rec_10y', 'total_inq', 'log_avail_bc', 'total_trades', 'never_delinq_pct', 'credit_hist_yrs', 'curr_lines', 'prior_prosper_loans']
    miss = f[CORE + BUREAU + ['rate', 'grade_ord']].isna().groupby(f['year']).mean().T.round(2); print("\nMissing share by year:"); print(miss.to_string())
    drop = [c for c in CORE + BUREAU if f.loc[f.year <= 2008, c].isna().mean() > 0.5]; CORE = [c for c in CORE if c not in drop]; BUREAU = [c for c in BUREAU if c not in drop]; print("dropped (mostly missing in training years):", drop)
    FS = {'P0 with lender rate': (CORE + ['rate'], ['employment', 'purpose']), 'P1 no lender score': (CORE, ['employment', 'purpose']),
          'P3 applicant-only rich': (CORE + BUREAU, ['employment', 'purpose', 'state']), 'P4 everything': (CORE + BUREAU + ['rate', 'grade_ord'], ['employment', 'purpose', 'state'])}
    Xc = {k: pd.get_dummies(f[n + c], columns=c, drop_first=True) for k, (n, c) in FS.items()}
    y, yr, tm, pi, fund, roi = f['default'].values, f['year'].values, f['tm'].values, f['profit'].values, f['funded'].values, f['roi'].values
    T1 = f.groupby('year').agg(N=('default', 'size'), default_rate=('default', 'mean'), mean_rate=('rate', 'mean'), mean_fico=('fico', 'mean'), mean_dti=('dti', 'mean'), mean_amount=('funded', 'mean'),
                               profit_per_loan=('profit', 'mean'), return_on_funded=('profit', lambda s: s.sum() / f.loc[s.index, 'funded'].sum())).reset_index(); save(T1, 'P1_data_summary_by_year'); print(); print(T1.round(3).to_string(index=False))
    g = f.groupby('default').apply(lambda x: x['profit'].sum() / x['funded'].sum()); print(f"\nprofit per $ funded: repaid {g[0]:+.3f}, default {g[1]:+.3f} -> break-even default probability ~ {g[0] / (g[0] - g[1]):.1%}")
    save(pd.DataFrame([dict(profit_per_dollar_repaid=g[0], profit_per_dollar_default=g[1], breakeven_default_prob=g[0] / (g[0] - g[1]))]), 'P2_breakeven')
    lag = ((f['reveal'] - f['tm']))
    desc = pd.DataFrame({'outcome': np.where(y == 1, 'default', 'repaid'), 'months_to_label': lag}).groupby('outcome')['months_to_label'].describe(percentiles=[.1, .25, .5, .75, .9]).reset_index(); save(desc, 'P3_label_lag_by_outcome'); print("\nMonths from origination to label known (exact ClosedDate):"); print(desc.round(1).to_string(index=False))

In [ ]:
# ---------- 1. model zoo and temporal designs ----------
LGB_REG = dict(n_estimators=250, learning_rate=0.03, num_leaves=15, max_depth=4, min_child_samples=100, subsample=0.8, subsample_freq=1, colsample_bytree=0.7, reg_lambda=10.0, random_state=SEED, verbose=-1)
MODELS = ['LGBM default', 'LGBM regularised', 'Logistic (L2)']
def fit_predict(kind, X, mtr):
    if kind == 'LGBM default': m = lgb.LGBMClassifier(n_estimators=150, learning_rate=0.05, max_depth=5, random_state=SEED, verbose=-1)
    elif kind == 'LGBM regularised': m = lgb.LGBMClassifier(**LGB_REG)
    else:
        Xf = X.astype('float32'); miss_ = [c for c in Xf.columns if Xf.loc[mtr, c].isna().any()]; ind = Xf[miss_].isna().astype('float32').add_suffix('_na')
        Xf = pd.concat([Xf.fillna(Xf.loc[mtr].median().fillna(0)), ind], axis=1); mu, sd = Xf.loc[mtr].mean().fillna(0), Xf.loc[mtr].std().replace(0, 1).fillna(1); Z = np.nan_to_num(((Xf - mu) / sd).values)
        return LogisticRegression(C=0.1, max_iter=3000).fit(Z[mtr], y[mtr]).predict_proba(Z)[:, 1]
    return m.fit(X[mtr], y[mtr]).predict_proba(X)[:, 1]
def fit_reg(X, target, mtr): return lgb.LGBMRegressor(**LGB_REG).fit(X[mtr], target[mtr]).predict(X)
DESIGNS = {'B': dict(label='B: train 2006-08, calibrate 2009, test 2010-11Q1', train_end=2008, ca=2009, tests=[2010, 2011]),
           'A': dict(label='A: train 2006-07, calibrate 2008, test 2009-11Q1', train_end=2007, ca=2008, tests=[2009, 2010, 2011])}
tlabel = lambda yy: '2011Q1' if yy == 2011 else str(yy)


In [ ]:
# ---------- Lending Club frame ----------
AUD = []
def audit(part, item, value, note=''): AUD.append(dict(part=part, item=item, value=value, note=note))
# ---- If the automatic search below cannot find your files, paste their FULL paths here (keep the quotes), e.g.
#      MANUAL_WIDE = '/Users/yourname/Downloads/lending_club_36m_wide.csv'
MANUAL_WIDE = ''
MANUAL_PAYMENTS = ''
MANUAL_LABEL_DATES = ''   # extra_data/label_dates.csv written by Final_Analysis.ipynb (optional)
HAS_LC = False; HAS_LCPAY = False; HAS_LCLAB = False
def find_file(patterns, extra_roots=()):
    for root in [Q2, *extra_roots, Q2.parent, Path.home()/'Downloads', Path.home()/'Documents']:
        if not Path(root).exists(): continue
        for pat in patterns:
            hits = [h for h in sorted(Path(root).rglob(pat)) if h.is_file()]
            if hits: return hits[0]
    return None
with section('LC load'):
    WIDE = Path(MANUAL_WIDE) if MANUAL_WIDE else find_file(['lending_club_36m_wide*.csv'])
    if WIDE is None: raise FileNotFoundError("lending_club_36m_wide.csv not found under Desktop/Q2, Desktop, Downloads or Documents. Put it in Desktop/Q2/extra_data/ (it is the file written by Extract_Extra_Data.ipynb) and rerun.")
    print("Lending Club wide file:", WIDE)
    W = pd.read_csv(WIDE, dtype={'id': str}); LC = W[W.year <= 2015].reset_index(drop=True)
    PAYF = Path(MANUAL_PAYMENTS) if MANUAL_PAYMENTS else find_file(['payments*.csv']); HAS_LCPAY = False
    if PAYF is not None:
        _p = pd.read_csv(PAYF, dtype={'id': str})
        if {'id', 'total_pymnt'} <= set(_p.columns):
            LC = LC.merge(_p[['id', 'total_pymnt']].drop_duplicates('id'), on='id', how='left'); LC['profit'] = LC['total_pymnt'] - LC['funded_amnt']; HAS_LCPAY = True; print("payments file:", PAYF)
    if not HAS_LCPAY: print("!! payments.csv not found or lacks id/total_pymnt: the Lending Club profit checks will be skipped (everything else still runs)")
    LABF = Path(MANUAL_LABEL_DATES) if MANUAL_LABEL_DATES else find_file(['label_dates*.csv'])
    if LABF is not None:
        _l = pd.read_csv(LABF, dtype={'id': str})
        if {'id', 'last_pymnt_d'} <= set(_l.columns):
            LC = LC.merge(_l[['id', 'last_pymnt_d']].drop_duplicates('id'), on='id', how='left'); HAS_LCLAB = True; print("label dates file:", LABF)
    if not HAS_LCLAB: print("!! label_dates.csv not found: the Lending Club label-lag sweep and real-time monitoring gap will be skipped")
    yL, yrL = LC['default'].values, LC['year'].values
    LC_NUM = ['loan_amnt', 'annual_inc', 'dti', 'fico_score', 'revol_util', 'delinq_2yrs', 'emp_length_yrs', 'int_rate']; LC_CAT = ['home_ownership', 'verification_status', 'purpose']
    XL = pd.get_dummies(LC[LC_NUM + LC_CAT], columns=LC_CAT, drop_first=True); XL1 = XL.drop(columns=['int_rate'])
    def fit_lgb(kind, X, mtr, seed=42):
        if kind == 'default': m = lgb.LGBMClassifier(n_estimators=150, learning_rate=0.05, max_depth=5, random_state=seed, verbose=-1)
        else: m = lgb.LGBMClassifier(**{**LGB_REG, 'random_state': seed, 'min_child_samples': 200})
        return m.fit(X[mtr], yL[mtr]).predict_proba(X)[:, 1]
    HAS_LC = True; print(f"Lending Club frame: {len(LC):,} loans; payments available: {HAS_LCPAY}")



In [ ]:
# ---------- stop here with a clear message if the Lending Club data were not loaded ----------
if not HAS_LC:
    raise RuntimeError("STOP: the Lending Club file was not loaded (see the message printed under 'LC load' above). Fix: put lending_club_36m_wide.csv in Desktop/Q2/extra_data/, or set MANUAL_WIDE = '/full/path/lending_club_36m_wide.csv' near the top of the previous cell, then run all again. Nothing below can run without it.")
print('Lending Club data loaded - continuing.')


In [ ]:
# ---------- 3. label timing: exact arrival dates, uniform delays, ACI, rolling recalibration (design B and A) ----------
def aci_reveal(p, y_, month, reveal, cal_scores, alpha, gamma, clip=(0.01, 0.99)):
    s = np.sort(cal_scores); n = len(s); T = len(y_); a = float(alpha); lo_c, hi_c = clip
    err = np.zeros(T); size = np.zeros(T); order = np.argsort(reveal, kind='stable')
    rv = np.asarray(reveal, float); mo = np.asarray(month, float); pp = np.asarray(p, float); yy = np.asarray(y_, int); ptr = 0
    ordl = order.tolist(); rvl = rv[order].tolist(); mol = mo.tolist(); pl = pp.tolist(); yl = yy.tolist(); sl = s.tolist(); errl = [0.0] * T; sizel = [0] * T
    for t in range(T):
        mt = mol[t]
        while ptr < T and rvl[ptr] <= mt and ordl[ptr] < t:
            a = min(max(a + gamma * (alpha - errl[ordl[ptr]]), lo_c), hi_c); ptr += 1
        k = int(np.ceil((n + 1) * (1 - a))); q = -np.inf if k <= 0 else (np.inf if k > n else sl[k - 1])
        in0 = pl[t] <= q; in1 = (1 - pl[t]) <= q; errl[t] = 1.0 - float(in1 if yl[t] == 1 else in0); sizel[t] = int(in0) + int(in1)
    err = np.array(errl); size = np.array(sizel, float)
    return 1 - err, size
def recal_reveal(p, y_, month, reveal, is_test, pool_ok, alpha, window, cal_scores, min_pool=200, horizon=36):
    sc = true_scores(p, y_); T = len(y_); cov = np.full(T, np.nan); size = np.full(T, np.nan)
    for m in np.unique(month[is_test]):
        idx = np.where(is_test & (month == m))[0]; pool = sc[pool_ok & (reveal <= m) & (month >= m - (horizon if window is None else window))]
        if len(pool) < min_pool: pool = cal_scores
        q = conf_q(pool, alpha); S = np.column_stack([p[idx] <= q, (1 - p[idx]) <= q]); cov[idx] = S[np.arange(len(idx)), y_[idx]]; size[idx] = S.sum(1)
    return cov[is_test], size[is_test]


In [ ]:
# ---------- shared: Lending Club feature sets, generic fitting, predictions for every model/feature set ----------
LC['grade_ord'] = LC['grade'].map({g: i for i, g in enumerate('ABCDEFG')}); LC['subgrade_ord'] = LC['sub_grade'].map({f"{g}{k}": 5 * i + k - 1 for i, g in enumerate('ABCDEFG') for k in range(1, 6)})
L_NUM = ['loan_amnt', 'annual_inc', 'dti', 'fico_score', 'revol_util', 'delinq_2yrs', 'emp_length_yrs']; L_CAT = ['home_ownership', 'verification_status', 'purpose']
L_BUR = ['open_acc', 'pub_rec', 'total_acc', 'inq_last_6mths', 'mths_since_last_delinq', 'pub_rec_bankruptcies', 'revol_bal', 'credit_hist_yrs']
FSL = {'F0': (L_NUM + ['int_rate'], L_CAT), 'F1': (L_NUM, L_CAT), 'F3': (L_NUM + L_BUR, L_CAT + ['addr_state']), 'F4': (L_NUM + L_BUR + ['int_rate', 'grade_ord', 'subgrade_ord', 'installment'], L_CAT + ['addr_state'])}
XLc = {k: pd.get_dummies(LC[n + c], columns=c, drop_first=True) for k, (n, c) in FSL.items()}
MODELS = ['LGBM default', 'LGBM regularised', 'Logistic (L2)']
def fit_gen(kind, X, yt, mtr, mcs=100, seed=SEED):
    if kind == 'LGBM default': m = lgb.LGBMClassifier(n_estimators=150, learning_rate=0.05, max_depth=5, random_state=seed, verbose=-1)
    elif kind == 'LGBM regularised': m = lgb.LGBMClassifier(**{**LGB_REG, 'random_state': seed, 'min_child_samples': mcs})
    else:
        Xf = X.astype('float32'); mi = [c for c in Xf.columns if Xf.loc[mtr, c].isna().any()]; ind = Xf[mi].isna().astype('float32').add_suffix('_na')
        Xf = pd.concat([Xf.fillna(Xf.loc[mtr].median().fillna(0)), ind], axis=1); mu, sd = Xf.loc[mtr].mean().fillna(0), Xf.loc[mtr].std().replace(0, 1).fillna(1); Z = np.nan_to_num(((Xf - mu) / sd).values)
        return LogisticRegression(C=0.1, max_iter=3000).fit(Z[mtr], yt[mtr]).predict_proba(Z)[:, 1]
    return m.fit(X[mtr], yt[mtr]).predict_proba(X)[:, 1]
monL = LC['month'].values; tmL = (LC['year'].values - 2007) * 12 + monL
P_L = {}; P_P = {}
with section('models'):
    for fs in FSL:
        for kind in MODELS: P_L[(fs, kind)] = fit_gen(kind, XLc[fs], yL, yrL <= 2011, mcs=200)
    PDES = {'B': dict(ca=2009, tr=2008, tests=[2010, 2011]), 'A': dict(ca=2008, tr=2007, tests=[2009, 2010, 2011])}
    for dk, v in PDES.items():
        for fs in Xc:
            for kind in MODELS: P_P[(dk, fs, kind)] = fit_gen(kind, Xc[fs], y, yr <= v['tr'], mcs=100)
    BASE_L = ('F0', 'LGBM default'); SEL_L = ('F3', 'LGBM regularised'); BASE_P = ('P0 with lender rate', 'LGBM default')
    SEL_P = {'B': ('P3 applicant-only rich', 'LGBM regularised'), 'A': ('P4 everything', 'LGBM regularised')}
    print("models fitted:", len(P_L), "Lending Club,", len(P_P), "Prosper")
def hits_static(p, yy, ca):
    sc = true_scores(p, yy); q = conf_q(sc[ca], ALPHA); return sc <= q, sc, q
RES = {}

In [ ]:
# ---------- R1: rolling-origin training years (item 1) ----------
with section('R1 no leakage in the calibration-2011 row'):
    rows = []
    for Cy in (2011, 2012, 2013):
        mtr = yrL <= Cy - 1; p = fit_gen('LGBM default', XLc['F0'], yL, mtr); hit, sc, q = hits_static(p, yL, yrL == Cy)
        rows.append(dict(calibration_year=Cy, model_trained_on=f"{LC.loc[mtr, 'year'].min()}-{LC.loc[mtr, 'year'].max()}", n_train=int(mtr.sum()), overlap_train_calibration=int((mtr & (yrL == Cy)).sum()),
                         **{f'cov_{t}': hit[yrL == t].mean() for t in range(Cy + 1, 2016)}))
    # for contrast: the LEAKY variant (model trained through the calibration year)
    p = fit_gen('LGBM default', XLc['F0'], yL, yrL <= 2011); hit, sc, q = hits_static(p, yL, yrL == 2011)
    rows.append(dict(calibration_year='2011 (LEAKY contrast: model trained through 2011)', model_trained_on='2007-2011', n_train=int((yrL <= 2011).sum()), overlap_train_calibration=int(((yrL <= 2011) & (yrL == 2011)).sum()), **{f'cov_{t}': hit[yrL == t].mean() for t in range(2012, 2016)}))
    R1 = pd.DataFrame(rows); save(R1, 'R1_rolling_origin_training_years'); print(R1.round(4).to_string(index=False)); print("\nTable 3 uses the first three rows: the model is trained strictly before the calibration year (overlap = 0).")

In [ ]:
# ---------- R2: coverage error vs test gap vs CHANGE in gap (item 2) ----------
with section('R2 gap change'):
    rows = []
    def add_cfg(port, design, model, p, yy, yrs, ca_year, tests):
        ca = yrs == ca_year; hit, sc, q = hits_static(p, yy, ca); gcal = 100 * (yy[ca].mean() - p[ca].mean())
        for t in tests:
            m = yrs == t; rows.append(dict(portfolio=port, design=design, model=model, test=str(t), g_cal=gcal, g_test=100 * (yy[m].mean() - p[m].mean()), cov_err=100 * (hit[m].mean() - 0.9)))
    for fs in FSL:
        for kind in MODELS: add_cfg('Lending Club', 'cal 2012', f'{fs}|{kind}', P_L[(fs, kind)], yL, yrL, 2012, (2013, 2014, 2015))
    for Cy in (2010, 2011, 2013):
        for fs in ('F0', 'F1'): add_cfg('Lending Club', f'cal {Cy}', f'{fs}|LGBM default', fit_gen('LGBM default', XLc[fs], yL, yrL <= Cy - 1), yL, yrL, Cy, range(Cy + 1, 2016))
    for dk, v in PDES.items():
        for fs in Xc:
            for kind in MODELS: add_cfg('Prosper', f'design {dk}', f'{fs}|{kind}', P_P[(dk, fs, kind)], y, yr, v['ca'], v['tests'])
    D = pd.DataFrame(rows); D['d_gap'] = D['g_test'] - D['g_cal']; save(D, 'R2_gap_configs')
    def ols(df, cols):
        X = sm.add_constant(df[cols]); r = sm.OLS(df['cov_err'], X).fit(); return r
    out = []
    for lab, sub in (('pooled', D), ('Lending Club', D[D.portfolio == 'Lending Club']), ('Prosper', D[D.portfolio == 'Prosper'])):
        for spec, cols in (('test gap only', ['g_test']), ('change in gap only', ['d_gap']), ('both', ['g_test', 'g_cal'])):
            r = ols(sub, cols); rng = np.random.default_rng(SEED); cl = sub.groupby(['portfolio', 'design']).ngroup().values; K = cl.max() + 1; bs = []
            for b in range(1000):
                pick = rng.integers(0, K, K); idx = np.concatenate([np.where(cl == c_)[0] for c_ in pick])
                try: bs.append(ols(sub.iloc[idx], cols).params.values)
                except Exception: pass
            bs = np.array(bs); out.append(dict(sample=lab, spec=spec, n=len(sub), R2=r.rsquared, adj_R2=r.rsquared_adj, **{f'b_{c_}': r.params[c_] for c_ in cols}, **{f'b_{c_}_ci': f"[{np.percentile(bs[:, i + 1], 2.5):.2f}, {np.percentile(bs[:, i + 1], 97.5):.2f}]" for i, c_ in enumerate(cols)}))
    R2 = pd.DataFrame(out); save(R2, 'R2_regressions'); print(R2.round(3).to_string(index=False))
    print("\nCalibration-year gap by portfolio/design (mean over models):"); print(D.groupby(['portfolio', 'design'])[['g_cal', 'g_test', 'd_gap', 'cov_err']].mean().round(2).to_string())
    fig, ax = plt.subplots(1, 2, figsize=(10.5, 4.2), sharey=True)
    for a_, xc, xl in ((ax[0], 'g_test', 'actual minus predicted default, TEST period (pts)'), (ax[1], 'd_gap', 'CHANGE in gap: test period minus calibration year (pts)')):
        for port, c_ in (('Lending Club', '#1f77b4'), ('Prosper', '#d62728')): s_ = D[D.portfolio == port]; a_.scatter(s_[xc], s_['cov_err'], s=14, c=c_, alpha=0.8, label=port)
        a_.axhline(0, c='k', lw=0.6); a_.axvline(0, c='k', lw=0.6); a_.set_xlabel(xl)
    ax[0].set_ylabel('static coverage minus 0.90 (pts)'); ax[0].legend(frameon=False); ax[0].set_title(f"(a) R2 = {out[0]['R2']:.2f}", fontsize=9); ax[1].set_title(f"(b) R2 = {out[1]['R2']:.2f}", fontsize=9); plt.tight_layout(); savefig('R2_gap_vs_change_in_gap'); plt.show()

In [ ]:
# ---------- R3: calibration age vs calendar drift (item 3): test-month fixed effects across calibration years ----------
with section('R3 age effect with month fixed effects'):
    def monthly_cov(p, yy, yrs, mons, ca_year, last_year):
        ca = yrs == ca_year; hit, sc, q = hits_static(p, yy, ca); key = yrs * 100 + mons; out = []
        for k in np.unique(key[(yrs > ca_year) & (yrs <= last_year)]):
            m = key == k; out.append((int(k), 100 * (hit[m].mean() - 0.9)))
        return out
    long = []
    for Cy in (2010, 2011, 2012, 2013):
        p = fit_gen('LGBM default', XLc['F0'], yL, yrL <= Cy - 1)
        for k, dv in monthly_cov(p, yL, LC['year'].values, monL, Cy, 2015): long.append(dict(portfolio='Lending Club', calib=Cy, month=k, cov_dev=dv, age_months=(k // 100 - Cy) * 12 + (k % 100)))
    fP = f; yrP, monP = f['year'].values, f['month'].values
    for Cy in (2007, 2008, 2009):
        p = fit_gen('LGBM default', Xc['P0 with lender rate'], y, yr <= Cy - 1)
        for k, dv in monthly_cov(p, y, yrP, monP, Cy, 2011): long.append(dict(portfolio='Prosper', calib=Cy, month=k, cov_dev=dv, age_months=(k // 100 - Cy) * 12 + (k % 100)))
    # variant 2: ONE model (trained before the earliest calibration year), several calibration years -> only calibration age differs within a test month
    pfix = fit_gen('LGBM default', XLc['F0'], yL, yrL <= 2009)
    for Cy in (2010, 2011, 2012, 2013):
        for k, dv in monthly_cov(pfix, yL, LC['year'].values, monL, Cy, 2015): long.append(dict(portfolio='Lending Club (fixed model 2007-09)', calib=Cy, month=k, cov_dev=dv, age_months=(k // 100 - Cy) * 12 + (k % 100)))
    pfixP = fit_gen('LGBM default', Xc['P0 with lender rate'], y, yr <= 2006)
    for Cy in (2007, 2008, 2009):
        for k, dv in monthly_cov(pfixP, y, yrP, monP, Cy, 2011): long.append(dict(portfolio='Prosper (fixed model 2006)', calib=Cy, month=k, cov_dev=dv, age_months=(k // 100 - Cy) * 12 + (k % 100)))
    LG = pd.DataFrame(long); save(LG, 'R3_monthly_coverage_by_calibration_year'); rows = []
    for port, sub in LG.groupby('portfolio'):
        sub = sub[sub.groupby('month')['calib'].transform('nunique') >= 2].copy(); sub['age_y'] = sub['age_months'] / 12
        def fe_slope(s):
            x = s['age_y'] - s.groupby('month')['age_y'].transform('mean'); yv = s['cov_dev'] - s.groupby('month')['cov_dev'].transform('mean'); return float((x * yv).sum() / (x * x).sum())
        b_fe = fe_slope(sub); months = sub['month'].unique(); rng = np.random.default_rng(SEED); bs = []
        for b in range(2000):
            pick = rng.choice(months, len(months)); s2 = pd.concat([sub[sub.month == m_].assign(month=f'{m_}_{i}') for i, m_ in enumerate(pick)]); bs.append(fe_slope(s2))
        # naive slope: one calibration year, calendar time and age move together
        one = sub[sub.calib == sub.calib.min()]; naive = np.polyfit(one['age_y'], one['cov_dev'], 1)[0]
        rows.append(dict(portfolio=port, n_obs=len(sub), n_months=len(months), calib_years=sorted(sub.calib.unique().tolist()), slope_pts_per_year_with_month_FE=b_fe, ci_lo=np.percentile(bs, 2.5), ci_hi=np.percentile(bs, 97.5), naive_slope_single_calibration=naive))
    R3 = pd.DataFrame(rows); save(R3, 'R3_age_effect_month_fe'); print("Coverage minus 0.90 (pts) per year of calibration age, holding the TEST MONTH fixed (several calibration years per month):"); print(R3.round(3).to_string(index=False))

In [ ]:
# ---------- R4: permutation test of exchangeability (item 4) ----------
def perm_test(sc_cal, sc_test, B, seed=SEED):
    pool = np.concatenate([sc_cal, sc_test]); n = len(sc_cal); N = len(pool); rng = np.random.default_rng(seed); k = int(np.ceil((n + 1) * (1 - ALPHA))); obs_q = conf_q(sc_cal, ALPHA); obs = float(np.mean(sc_test <= obs_q)); null = np.empty(B)
    for b in range(B):
        perm = rng.permutation(N); cal = pool[perm[:n]]; tst = pool[perm[n:]]; q = np.partition(cal, k - 1)[k - 1] if k <= n else np.inf; null[b] = np.mean(tst <= q)
    m0 = null.mean(); p = (1 + np.sum(np.abs(null - m0) >= abs(obs - m0))) / (B + 1); return obs, m0, np.percentile(null, [2.5, 97.5]), p
with section('R4 permutation tests'):
    rows = []
    for pname, key in (('baseline', BASE_L), ('selected', SEL_L)):
        p = P_L[key]; sc = true_scores(p, yL)
        for t in (2013, 2014, 2015):
            obs, m0, (lo, hi), pv = perm_test(sc[yrL == 2012], sc[yrL == t], 1000); rows.append(dict(portfolio='Lending Club', design='cal 2012', pipeline=pname, test=str(t), n_cal=int((yrL == 2012).sum()), n_test=int((yrL == t).sum()), observed=obs, null_mean=m0, null_lo=lo, null_hi=hi, p_value=pv))
    for dk, v in PDES.items():
        for pname, key in (('baseline', BASE_P), ('selected', SEL_P[dk])):
            p = P_P[(dk,) + key]; sc = true_scores(p, y)
            for t in v['tests']:
                obs, m0, (lo, hi), pv = perm_test(sc[yr == v['ca']], sc[yr == t], 5000); rows.append(dict(portfolio='Prosper', design=dk, pipeline=pname, test=tlabel(t), n_cal=int((yr == v['ca']).sum()), n_test=int((yr == t).sum()), observed=obs, null_mean=m0, null_lo=lo, null_hi=hi, p_value=pv))
    R4 = pd.DataFrame(rows); R4['p_holm'] = np.nan
    for port, ix in R4.groupby('portfolio').groups.items(): R4.loc[ix, 'p_holm'] = multipletests(R4.loc[ix, 'p_value'], method='holm')[1]
    save(R4, 'R4_permutation_tests'); print(R4.round(4).to_string(index=False))

In [ ]:
# ---------- R5-R7: block-bootstrap intervals (calibration resampling + month blocks), segments, FDR (items 5-7) ----------
def boot_cov(sc_cal, sc_test, month_test, segs=None, B=1000, seed=SEED, minN=150):
    rng = np.random.default_rng(seed); months = np.unique(month_test); M = len(months); mt = np.searchsorted(months, month_test); cells = {}
    allseg = {'ALL': np.full(len(sc_test), 'all')}; allseg.update(segs or {})
    for sname, lab in allseg.items():
        lab = np.asarray(lab).astype(str)
        for v in np.unique(lab):
            m = lab == v
            if m.sum() < minN and sname != 'ALL': continue
            cells[(sname, v)] = [np.sort(sc_test[m & (mt == j)]) for j in range(M)]
    keys = list(cells); n = np.array([[len(a) for a in cells[k]] for k in keys], float); q0 = conf_q(sc_cal, ALPHA)
    est = np.array([sum(np.searchsorted(a, q0, side='right') for a in cells[k]) / max(n[i].sum(), 1) for i, k in enumerate(keys)]); bs = np.empty((B, len(keys)))
    for b in range(B):
        qb = conf_q(rng.choice(sc_cal, len(sc_cal), replace=True), ALPHA); w = np.bincount(rng.integers(0, M, M), minlength=M).astype(float)
        H = np.array([[np.searchsorted(a, qb, side='right') for a in cells[k]] for k in keys], float); den = (n * w).sum(1); bs[b] = (H * w).sum(1) / np.where(den > 0, den, np.nan)
    lo, hi = np.nanpercentile(bs, [2.5, 97.5], axis=0); pv = np.minimum(1.0, 2 * np.minimum(np.nanmean(bs >= 0.9, axis=0), np.nanmean(bs <= 0.9, axis=0))); pv = np.maximum(pv, 1.0 / B)
    return pd.DataFrame(dict(segment=[k[0] for k in keys], value=[k[1] for k in keys], N=n.sum(1).astype(int), cov=est, ci_lo=lo, ci_hi=hi, p_boot=pv))
top = lambda s, k: np.where(pd.Series(np.asarray(s)).isin(pd.Series(np.asarray(s)).value_counts().index[:k]), np.asarray(s), 'other')
qb_ = lambda s, k, lab: pd.qcut(pd.Series(np.asarray(s, float)), k, labels=[f'{lab} Q{i+1}' for i in range(k)], duplicates='drop').astype(str).values
SEGS = {}
with section('R5-R7 bootstrap intervals and segments'):
    out = []; scL = true_scores(P_L[BASE_L], yL); teL = yrL >= 2013
    lc_segs = {'grade': LC['grade'].values, 'purpose': top(LC['purpose'].values, 7), 'home ownership': np.where(LC['home_ownership'].isin(['RENT', 'MORTGAGE', 'OWN']), LC['home_ownership'], 'other'), 'verification': LC['verification_status'].values,
               'income quartile': qb_(LC['annual_inc'], 4, 'income'), 'FICO band': pd.cut(LC['fico_score'], [0, 680, 700, 720, 740, 900], right=False).astype(str).values, 'loan amount tercile': qb_(LC['loan_amnt'], 3, 'amount'), 'DTI quartile': qb_(LC['dti'], 4, 'dti')}
    t = boot_cov(scL[yrL == 2012], scL[teL], (LC['year'].values * 100 + monL)[teL], {k: v[teL] for k, v in lc_segs.items()}, B=1000); t.insert(0, 'portfolio', 'Lending Club'); t.insert(1, 'design', '-'); out.append(t)
    gm = {0: 'AA', 1: 'A', 2: 'B', 3: 'C', 4: 'D', 5: 'E', 6: 'HR'}; ge = f['grade_ord'].map(gm).fillna('none').values
    for dk, v in PDES.items():
        te_ = np.isin(yr, v['tests']); scP = true_scores(P_P[(dk,) + BASE_P], y)
        ps = {'grade': ge, 'employment': f['employment'].values, 'homeowner': np.where(f['homeowner'] == 1, 'yes', 'no'), 'purpose': top(f['purpose'].values, 6), 'FICO band': pd.cut(f['fico'], [0, 660, 700, 740, 900], right=False).astype(str).values,
              'income quartile': qb_(f['StatedMonthlyIncome'], 4, 'income'), 'loan amount tercile': qb_(f['funded'], 3, 'amount')}
        t = boot_cov(scP[yr == v['ca']], scP[te_], (f['year'].values * 100 + f['month'].values)[te_], {k: a[te_] for k, a in ps.items()}, B=1000); t.insert(0, 'portfolio', 'Prosper'); t.insert(1, 'design', dk); out.append(t)
    SEG = pd.concat(out, ignore_index=True); SEG['p_bh'] = np.nan
    for (port, dk), ix in SEG[SEG.segment != 'ALL'].groupby(['portfolio', 'design']).groups.items(): SEG.loc[ix, 'p_bh'] = multipletests(SEG.loc[ix, 'p_boot'], method='fdr_bh')[1]
    SEG['sig_ci'] = (SEG.ci_lo > 0.9) | (SEG.ci_hi < 0.9); SEG['sig_bh'] = SEG['p_bh'] < 0.05; save(SEG, 'R5_segment_coverage_block_bootstrap')
    for (port, dk), g in SEG[SEG.segment != 'ALL'].groupby(['portfolio', 'design']): print(f"{port} {dk}: {len(g)} segments; 95% block-bootstrap CI excludes 0.90 in {int(g.sig_ci.sum())}; significant after Benjamini-Hochberg (5%) in {int(g.sig_bh.sum())}; coverage range {g['cov'].min():.3f}-{g['cov'].max():.3f}")
    print("\nLending Club by grade:"); print(SEG[(SEG.portfolio == 'Lending Club') & (SEG.segment == 'grade')].round(4).to_string(index=False))
    print("\nPer-period intervals for the Prosper validity table (calibration resampling + month blocks):"); rows = []
    for dk, v in PDES.items():
        for pname, key in (('baseline', BASE_P), ('selected', SEL_P[dk])):
            scP = true_scores(P_P[(dk,) + key], y)
            for t_ in v['tests']:
                m = yr == t_; r = boot_cov(scP[yr == v['ca']], scP[m], (f['year'].values * 100 + f['month'].values)[m], None, B=2000).iloc[0]; hl, hh = wilson(r['cov'] * m.sum(), m.sum())
                rows.append(dict(design=dk, pipeline=pname, period=tlabel(t_), N=int(m.sum()), cov=r['cov'], wilson_lo=hl, wilson_hi=hh, block_lo=r['ci_lo'], block_hi=r['ci_hi'], p_boot=r['p_boot']))
    R5P = pd.DataFrame(rows); save(R5P, 'R5_prosper_validity_block_intervals'); print(R5P.round(4).to_string(index=False))
    print("\nLending Club per test year (calibration resampling + month blocks):"); rows = []
    for t_ in (2013, 2014, 2015):
        m = yrL == t_; r = boot_cov(scL[yrL == 2012], scL[m], monL[m], None, B=2000).iloc[0]; rows.append(dict(test=t_, cov=r['cov'], block_lo=r['ci_lo'], block_hi=r['ci_hi'], p_boot=r['p_boot']))
    print(pd.DataFrame(rows).round(4).to_string(index=False))

In [ ]:
# ---------- R6: autocorrelation-robust slopes for the monthly monitoring regressions (item 6) ----------
def monthly_df(p, yy, yrs, mons, ca_year, tests):
    hit, sc, q = hits_static(p, yy, yrs == ca_year); key = yrs * 100 + mons; rows = []
    for k in np.unique(key[np.isin(yrs, tests)]):
        m = key == k; rows.append(dict(month=int(k), gap=100 * (yy[m].mean() - p[m].mean()), cov_dev=100 * (hit[m].mean() - 0.9)))
    d = pd.DataFrame(rows).sort_values('month').reset_index(drop=True); d['age_y'] = (np.arange(len(d)) + 1) / 12.0; return d
def mbb_slope(x, yv, L=6, B=2000, seed=SEED):
    rng = np.random.default_rng(seed); n = len(x); nb = int(np.ceil(n / L)); out = []
    for b in range(B):
        st = rng.integers(0, n - L + 1, nb); idx = np.concatenate([np.arange(s, s + L) for s in st])[:n]; out.append(np.polyfit(x[idx], yv[idx], 1)[0])
    return np.percentile(out, [2.5, 97.5])
with section('R6 HAC and block-bootstrap slopes'):
    sets = [('Lending Club', '-', monthly_df(P_L[BASE_L], yL, LC['year'].values, monL, 2012, (2013, 2014, 2015)))] + [('Prosper', dk, monthly_df(P_P[(dk,) + BASE_P], y, f['year'].values, f['month'].values, v['ca'], v['tests'])) for dk, v in PDES.items()]
    rows = []; MON = {}
    for port, dk, d in sets:
        MON[(port, dk)] = d
        for lab, xc in (('coverage error on monthly gap', 'gap'), ('coverage error on calibration age (per year)', 'age_y')):
            r = sm.OLS(d['cov_dev'], sm.add_constant(d[xc])).fit(cov_type='HAC', cov_kwds={'maxlags': 6}); lo_b, hi_b = mbb_slope(d[xc].values, d['cov_dev'].values)
            ci = r.conf_int().loc[xc]; rows.append(dict(portfolio=port, design=dk, regression=lab, months=len(d), slope=r.params[xc], hac_lo=ci[0], hac_hi=ci[1], block_boot_lo=lo_b, block_boot_hi=hi_b, corr=np.corrcoef(d[xc], d['cov_dev'])[0, 1]))
    R6 = pd.DataFrame(rows); save(R6, 'R6_autocorrelation_robust_slopes'); print(R6.round(3).to_string(index=False))

In [ ]:
# ---------- R8: is the validation-year model choice distinguishable from noise? (item 8) ----------
with section('R8 validation bootstrap'):
    rows = []; rng = np.random.default_rng(SEED)
    def val_study(port, label, getp, yy, yrs, val_year, months, sel_key, base_key, B=2000):
        keys = [(fs, kd) for fs in getp['fs'] for kd in MODELS]; v = yrs == val_year; ll = {}
        for k in keys:
            p = np.clip(getp['fit'](k, val_year), 1e-6, 1 - 1e-6); ll[k] = -(yy[v] * np.log(p[v]) + (1 - yy[v]) * np.log(1 - p[v]))
        mv = months[v]; um = np.unique(mv); idx_m = [np.where(mv == m_)[0] for m_ in um]; mean_ll = pd.Series({k: ll[k].mean() for k in keys}); best = mean_ll.idxmin(); wins = {k: 0 for k in keys}; diffs = []; rk = []
        for b in range(B):
            pick = rng.integers(0, len(um), len(um)); ii = np.concatenate([idx_m[j] for j in pick]); m_b = pd.Series({k: ll[k][ii].mean() for k in keys}); wins[m_b.idxmin()] += 1; diffs.append(m_b[sel_key] - m_b[base_key]); rk.append(int((m_b < m_b[sel_key]).sum()) + 1)
        return dict(portfolio=port, design=label, validation_year=val_year, chosen_by_validation=f"{best[0]} | {best[1]}", prespecified_selected=f"{sel_key[0]} | {sel_key[1]}", n_val=int(v.sum()), logloss_selected=mean_ll[sel_key], logloss_baseline=mean_ll[base_key],
                    diff_sel_minus_base=mean_ll[sel_key] - mean_ll[base_key], diff_ci=f"[{np.percentile(diffs, 2.5):.4f}, {np.percentile(diffs, 97.5):.4f}]", share_of_resamples_selected_is_best=wins[sel_key] / B, median_rank_of_selected=float(np.median(rk)), share_diff_below_zero=float(np.mean(np.array(diffs) < 0)))
    gl = dict(fs=list(FSL), fit=lambda k, vy: fit_gen(k[1], XLc[k[0]], yL, yrL <= vy - 1, mcs=200)); rows.append(val_study('Lending Club', 'val 2011', gl, yL, yrL, 2011, tmL, SEL_L, BASE_L))
    for dk, v in PDES.items():
        gp = dict(fs=list(Xc), fit=lambda k, vy: fit_gen(k[1], Xc[k[0]], y, yr <= vy - 1, mcs=100)); rows.append(val_study('Prosper', f'design {dk}', gp, y, yr, v['tr'], tm, SEL_P[dk], BASE_P))
    R8 = pd.DataFrame(rows); save(R8, 'R8_model_selection_bootstrap'); print(R8.round(4).T.to_string())

In [ ]:
# ---------- R9: delayed-feedback gain sweep (item 9), R10: Prosper label-lag sweep (item 10) ----------
GAINS = [0.01, 0.03, 0.1, 0.25, 0.5, 1.0, 2.0, 4.0]; SW = []
def sweep_set(port, p_all, yy, yrs, ms, ca_year, reveal_exact, label_mon):
    sel = np.where(yrs >= ca_year)[0]; p_s, ys, m_s, yrs_ = p_all[sel], yy[sel], ms[sel], yrs[sel]; ca_s, te_s = yrs_ == ca_year, yrs_ > ca_year; cal_sc = true_scores(p_s, ys)[ca_s]; NPM = pd.Series(m_s[te_s]).value_counts().mean()
    def cov_of(rv, g):
        c_, s_ = aci_reveal(p_s[te_s], ys[te_s], m_s[te_s], rv[te_s], cal_sc, ALPHA, g); last = yrs_[te_s] == yrs_[te_s].max(); return c_.mean(), c_[last].mean(), s_.mean()
    c0, s0 = None, None
    hit_static = (true_scores(p_s, ys) <= conf_q(cal_sc, ALPHA))[te_s]; SW.append(dict(portfolio=port, arrival='static', gain=np.nan, pooled=hit_static.mean(), last_year=hit_static[yrs_[te_s] == yrs_[te_s].max()].mean(), size=np.nan))
    for arr, rv in (('uniform 6 months', m_s + 6), ('uniform 12 months', m_s + 12)) + ((('exact arrival', reveal_exact[sel]),) if reveal_exact is not None else ()):
        for g in GAINS:
            a, b_, s_ = cov_of(rv, g / NPM); SW.append(dict(portfolio=port, arrival=arr, gain=g, pooled=a, last_year=b_, size=s_))
    a, b_, s_ = cov_of(m_s, 0.015); SW.append(dict(portfolio=port, arrival='immediate (gamma 0.015)', gain=np.nan, pooled=a, last_year=b_, size=s_))
with section('R9 gain sweep'):
    shf = LC.sample(frac=1.0, random_state=SEED); kp = shf[shf.groupby(['year', 'month']).cumcount() < 2500]; si = kp.sort_values('issue_d', kind='stable').index.values
    pfull = P_L[BASE_L]; sweep_set('Lending Club (2,500/month sample)', pfull[si], yL[si], LC['year'].values[si], tmL[si], 2012, None, None)
    selP = np.where(yr >= 2009)[0]; sweep_set('Prosper design B', P_P[('B',) + BASE_P], y, yr, tm, 2009, f['reveal'].values, None)
    SWD = pd.DataFrame(SW); save(SWD, 'R9_gain_sweep'); print(SWD.round(3).to_string(index=False))
    fig, ax = plt.subplots(1, 2, figsize=(10.5, 4), sharey=False)
    for a_, port in zip(ax, SWD.portfolio.unique()):
        s_ = SWD[SWD.portfolio == port]; base = s_[s_.arrival == 'static']['last_year'].iloc[0]
        for arr, g in s_[s_.gain.notna()].groupby('arrival'): a_.plot(g['gain'], g['last_year'], marker='o', label=arr)
        a_.axhline(base, c='gray', ls='--', label='static'); a_.axhline(0.9, c='k', ls=':'); a_.set_xscale('log'); a_.set_xlabel('batch gain (gamma x loans per month)'); a_.set_ylabel('coverage in the last test year'); a_.set_title(port, fontsize=9); a_.legend(fontsize=7, frameon=False)
    plt.tight_layout(); savefig('R9_gain_sweep'); plt.show()
with section('R10 label-lag sweep (Prosper, exact dates + extra lag on defaults)'):
    rows = []; NPMp = pd.Series(tm[yr >= 2010]).value_counts().mean(); p_all = P_P[('B',) + BASE_P]; sel = np.where(yr >= 2009)[0]; p_s, ys, m_s, yrs_ = p_all[sel], y[sel], tm[sel], yr[sel]; ca_s, te_s = yrs_ == 2009, yrs_ > 2009; cal_sc = true_scores(p_s, ys)[ca_s]
    for k in range(0, 9):
        rv = np.where(y == 1, np.maximum(f['reveal'].values + k, tm + 1), f['reveal'].values)[sel]
        c_, s_ = aci_reveal(p_s[te_s], ys[te_s], m_s[te_s], rv[te_s], cal_sc, ALPHA, 0.25 / NPMp); r_, rs_ = recal_reveal(p_s, ys, m_s, rv, te_s, yrs_ >= 2009, ALPHA, None, cal_sc)
        snap = (2010 - 2006) * 12 + 1; known = (rv <= snap) & (m_s <= snap) & (m_s >= snap - 36); allw = (m_s <= snap) & (m_s >= snap - 36)
        rows.append(dict(extra_lag_months=k, aci_gain025_cov=c_.mean(), rolling_known_labels_cov=r_.mean(), rolling_size=rs_.mean(), share_known_jan2010=known.sum() / max(allw.sum(), 1), default_rate_known_jan2010=ys[known].mean(), eventual_default_rate_jan2010=ys[allw].mean()))
    R10 = pd.DataFrame(rows); save(R10, 'R10_prosper_lag_sweep'); print(R10.round(4).to_string(index=False)); print("\nLending Club cannot be re-run here: its charge-off lag needs last_pymnt_d from the raw file; the Prosper sweep varies the same assumption with exact dates.")

In [ ]:
# ---------- R10b: Lending Club charge-off lag sweep, 2-8 months (item 10; needs label_dates.csv) ----------
with section('R10b Lending Club charge-off lag sweep'):
    if not HAS_LCLAB: raise RuntimeError('label_dates.csv not loaded (see LC load): run Final_Analysis.ipynb once, or set MANUAL_LABEL_DATES')
    ld = pd.to_datetime(LC['last_pymnt_d'], format='%b-%Y', errors='coerce'); lpm = ((ld.dt.year - 2007) * 12 + ld.dt.month).values.astype(float); missL = np.isnan(lpm)
    def reveal_lc(lag):
        r = np.where(yL == 1, lpm + lag, lpm); r = np.where(missL, tmL + np.where(yL == 1, 12, 36), r); return np.maximum(r, tmL + 1)
    shf = LC.sample(frac=1.0, random_state=SEED); kp = shf[shf.groupby(['year', 'month']).cumcount() < 2500]; si = kp.sort_values('issue_d', kind='stable').index.values
    si = si[LC['year'].values[si] >= 2012]; p0 = P_L[BASE_L][si]; ys_, ms_, yrs_ = yL[si], tmL[si], LC['year'].values[si]; ca_s, te_s = yrs_ == 2012, yrs_ >= 2013
    cal_sc = true_scores(p0, ys_)[ca_s]; NPM = pd.Series(ms_[te_s]).value_counts().mean(); last = yrs_[te_s] == 2015; rows = []; snaps = []
    for lag in range(2, 9):
        rv = reveal_lc(lag); rvs = rv[si]
        for g in (0.1, 0.25):
            c_, _ = aci_reveal(p0[te_s], ys_[te_s], ms_[te_s], rvs[te_s], cal_sc, ALPHA, g / NPM); rows.append(dict(charge_off_lag=lag, method=f'ACI batch gain {g}, realistic arrival', cov_2015=c_[last].mean(), pooled=c_.mean(), cov_defaulters=c_[ys_[te_s] == 1].mean()))
        r_, _ = recal_reveal(p0, ys_, ms_, rvs, te_s, yrs_ >= 2012, ALPHA, None, cal_sc, min_pool=300); rows.append(dict(charge_off_lag=lag, method='rolling, all known labels', cov_2015=r_[last].mean(), pooled=r_.mean(), cov_defaulters=r_[ys_[te_s] == 1].mean()))
        for sy in (2013, 2014, 2015):
            snap = (sy - 2007) * 12 + 12; win = (tmL <= snap) & (tmL > snap - 36); kn = win & (rv <= snap)
            snaps.append(dict(charge_off_lag=lag, snapshot=f'Dec {sy}', share_known=kn.sum() / win.sum(), default_rate_known=yL[kn].mean(), eventual_default_rate=yL[win].mean()))
        med = pd.Series(rv - tmL).groupby(yL).median(); snaps[-1].update(median_months_default=med.get(1), median_months_repaid=med.get(0))
    R10b = pd.DataFrame(rows); save(R10b, 'R10b_lc_lag_sweep_methods'); S10b = pd.DataFrame(snaps); save(S10b, 'R10b_lc_lag_sweep_label_selection')
    print(R10b.round(3).to_string(index=False)); print(); print(S10b.round(3).to_string(index=False))


In [ ]:
# ---------- R11: per-month sample vs all loans vs resamples (item 11) ----------
with section('R11 sampling check'):
    p0 = P_L[BASE_L]; rows = []
    def run_all(label, idx):
        p_s, ys, ms_, yrs_ = p0[idx], yL[idx], tmL[idx], LC['year'].values[idx]; ca_s, te_s = yrs_ == 2012, yrs_ >= 2013; cal_sc = true_scores(p_s, ys)[ca_s]; NPM = pd.Series(ms_[te_s]).value_counts().mean(); last = yrs_[te_s] == 2015
        hit = (true_scores(p_s, ys) <= conf_q(cal_sc, ALPHA))[te_s]; rows.append(dict(sample=label, method='static', pooled=hit.mean(), cov_2015=hit[last].mean()))
        c_, s_ = aci_reveal(p_s[te_s], ys[te_s], ms_[te_s], ms_[te_s], cal_sc, ALPHA, 0.015); rows.append(dict(sample=label, method='ACI gamma 0.015, immediate', pooled=c_.mean(), cov_2015=c_[last].mean()))
        for dly in (6, 12):
            for g in (0.1, 0.25):
                c_, s_ = aci_reveal(p_s[te_s], ys[te_s], ms_[te_s], (ms_ + dly)[te_s], cal_sc, ALPHA, g / NPM); rows.append(dict(sample=label, method=f'ACI batch gain {g}, delay {dly}', pooled=c_.mean(), cov_2015=c_[last].mean()))
            r_, _ = recal_reveal(p_s, ys, ms_, ms_ + dly, te_s, yrs_ >= 2012, ALPHA, 12, cal_sc, min_pool=300); rows.append(dict(sample=label, method=f'rolling recalibration, delay {dly}', pooled=r_.mean(), cov_2015=r_[last].mean()))
    run_all('all loans', np.where(LC['year'].values >= 2012)[0])
    for sd in range(5):
        shf = LC[LC.year >= 2012].sample(frac=1.0, random_state=sd); kp = shf[shf.groupby(['year', 'month']).cumcount() < 2500]; run_all(f'2,500/month sample, seed {sd}', kp.sort_values('issue_d', kind='stable').index.values)
    R11 = pd.DataFrame(rows); save(R11, 'R11_sampling_check'); piv = R11.pivot_table(index='method', columns='sample', values='cov_2015'); print("Coverage in 2015:"); print(piv.round(3).to_string()); piv2 = R11.pivot_table(index='method', columns='sample', values='pooled'); print("\nPooled 2013-2015:"); print(piv2.round(3).to_string())
    sm_cols = [c for c in piv.columns if c.startswith('2,500')]; print("\nSample seeds: min / max of 2015 coverage by method"); print(pd.DataFrame({'all': piv['all loans'], 'sample_min': piv[sm_cols].min(axis=1), 'sample_max': piv[sm_cols].max(axis=1)}).round(3).to_string())

In [ ]:
# ---------- R12: Lending Club with a 2,047-loan calibration set (item 12) ----------
with section('R12 calibration size'):
    sc = true_scores(P_L[BASE_L], yL); ca_idx = np.where(yrL == 2012)[0]; rng = np.random.default_rng(SEED); rows = []
    for t in (2013, 2014, 2015):
        ti = np.where(yrL == t)[0]; full_q = conf_q(sc[ca_idx], ALPHA); full_cov = (sc[ti] <= full_q).mean(); c1 = np.empty(1000); c2 = np.empty(1000)
        for b in range(1000):
            qs = conf_q(sc[rng.choice(ca_idx, 2047, replace=False)], ALPHA); c1[b] = (sc[ti] <= qs).mean(); c2[b] = (sc[rng.choice(ti, 5599, replace=False)] <= qs).mean() if t > 2013 else (sc[ti] <= qs).mean()
        rows.append(dict(test=t, coverage_full_calibration=full_cov, n2047_mean=c1.mean(), n2047_sd=c1.std(), n2047_lo=np.percentile(c1, 2.5), n2047_hi=np.percentile(c1, 97.5), plus_test_5599_sd=c2.std(), plus_test_5599_lo=np.percentile(c2, 2.5), plus_test_5599_hi=np.percentile(c2, 97.5)))
    R12 = pd.DataFrame(rows); save(R12, 'R12_calibration_size'); print(R12.round(4).to_string(index=False)); print("\nProsper design B (calibration 2,047 loans), month-block + calibration bootstrap 95% intervals: see R5_prosper_validity_block_intervals.")

In [ ]:
# ---------- R13: monitoring thresholds, precision/recall, real-time gap (item 13) ----------
with section('R13 monitoring sweep'):
    rows = []; TH = [0.5, 1.0, 1.5, 2.0, 3.0, 4.0]
    def evaluate(port, dk, gap_series, d, label):
        dd = d.assign(gap_rt=gap_series).dropna(subset=['gap_rt']).copy(); dd['gap_trail'] = dd['gap_rt'].rolling(3, min_periods=3).mean(); dd['cov_trail'] = dd['cov_dev'].rolling(3, min_periods=3).mean(); dd = dd.dropna(subset=['gap_trail', 'cov_trail'])
        bad = dd['cov_trail'].abs() > 1.0
        for th in TH:
            al = dd['gap_trail'].abs() > th; tp = int((al & bad).sum()); fp = int((al & ~bad).sum()); fn = int((~al & bad).sum())
            rows.append(dict(portfolio=port, design=dk, gap_version=label, threshold_pts=th, windows=len(dd), coverage_off_windows=int(bad.sum()), alarms=int(al.sum()), precision=tp / max(tp + fp, 1), recall=tp / max(tp + fn, 1)))
    for (port, dk), d in MON.items():
        evaluate(port, dk, d['gap'], d, 'eventual gap (all labels, hindsight)')
        if port == 'Prosper':
            v = PDES[dk]; p = P_P[(dk,) + BASE_P]; rv = f['reveal'].values; rt = []
            for k in d['month']:
                m_idx = (k // 100 - 2006) * 12 + (k % 100); win = (tm >= m_idx - 12) & (tm <= m_idx - 1); known = win & (rv <= m_idx)
                rt.append(100 * (y[known].mean() - p[known].mean()) if known.sum() >= 100 else np.nan)
            evaluate(port, dk, pd.Series(rt, index=d.index), d, 'real-time gap (labels known at the time)')
    if HAS_LCLAB:
        d = MON[('Lending Club', '-')]; rv = reveal_lc(4); pL = P_L[BASE_L]; rt = []
        for k in d['month']:
            m_idx = (k // 100 - 2007) * 12 + (k % 100); win = (tmL >= m_idx - 12) & (tmL <= m_idx - 1); known = win & (rv <= m_idx)
            rt.append(100 * (yL[known].mean() - pL[known].mean()) if known.sum() >= 100 else np.nan)
        evaluate('Lending Club', '-', pd.Series(rt, index=d.index), d, 'real-time gap (labels known at the time)')
    R13 = pd.DataFrame(rows); save(R13, 'R13_monitoring_threshold_sweep'); print(R13.round(3).to_string(index=False)); print("" if HAS_LCLAB else "\nLending Club real-time version skipped: label_dates.csv not loaded.")
    fig, ax = plt.subplots(figsize=(5.5, 4.2))
    for (port, dk, ver), g in R13.groupby(['portfolio', 'design', 'gap_version']): ax.plot(g['recall'], g['precision'], marker='o', ms=3, label=f"{port} {dk} {'RT' if ver.startswith('real') else 'hindsight'}".replace(' -', ''))
    ax.set_xlabel('recall'); ax.set_ylabel('precision'); ax.legend(fontsize=6, frameon=False); ax.set_title('alarm on trailing 3-month gap, thresholds 0.5-4 pts', fontsize=9); plt.tight_layout(); savefig('R13_precision_recall'); plt.show()

In [ ]:
# ---------- R14: reviewer break-even with separate sensitivity and specificity (item 14) ----------
with section('R14 break-even with separate accuracies'):
    rows = []; SR = [0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
    def frontier(port, dk, label, R, P0, P1, Fr, derived=False):
        for cost in (0.0, 0.01):
            for sr in SR:
                sd = (R + (1 - sr) * P0 + cost * Fr) / (-P1); rows.append(dict(portfolio=port, design=dk, policy=label, review_cost=cost, accuracy_on_repayers=sr, breakeven_accuracy_on_defaulters=sd, derived_from_reported_gains=derived))
    for dk, v in PDES.items():
        te = np.isin(yr, v['tests']); N = te.sum(); S = conformal_sets(P_P[(dk,) + BASE_P], y, yr == v['ca'])
        for lab, SS in S.items():
            ap, rv, rj = tiers(SS); R = pi[te & rj].sum() / N; P0 = pi[te & rv & (y == 0)].sum() / N; P1 = pi[te & rv & (y == 1)].sum() / N; Fr = fund[te & rv].sum() / N
            frontier('Prosper', dk, lab, R, P0, P1, Fr); rows.append(dict(portfolio='Prosper', design=dk, policy=lab, review_cost=np.nan, accuracy_on_repayers=np.nan, breakeven_accuracy_on_defaulters=np.nan, derived_from_reported_gains=False, note=f"R={R:.1f} P0={P0:.1f} P1={P1:.1f} F={Fr:.0f} per applicant"))
    # Lending Club: loan-level profit unavailable here; per-applicant sums derived from the gains reported in Table 8 / the break-even text
    if HAS_LCPAY:
        teL_ = yrL >= 2013; NL = teL_.sum(); piL = LC['profit'].values; fuL = LC['funded_amnt'].values.astype(float); SL = conformal_sets(P_L[BASE_L], yL, yrL == 2012)
        for lab, SS in SL.items():
            ap, rv_, rj = tiers(SS); R = piL[teL_ & rj].sum() / NL; P0 = piL[teL_ & rv_ & (yL == 0)].sum() / NL; P1 = piL[teL_ & rv_ & (yL == 1)].sum() / NL; Fr = fuL[teL_ & rv_].sum() / NL
            frontier('Lending Club', '-', lab, R, P0, P1, Fr); rows.append(dict(portfolio='Lending Club', design='-', policy=lab, note=f"R={R:.1f} P0={P0:.1f} P1={P1:.1f} F={Fr:.0f} per applicant"))
    else:
        # fallback: per-applicant sums derived from the gains reported in Table 8 / the break-even text
        frontier('Lending Club', '-', 'static', 0.0, 259.1, -139.66, 1552.0, True)
    R14 = pd.DataFrame(rows); save(R14, 'R14_breakeven_sensitivity_specificity')
    print(R14[R14.review_cost == 0.0][['portfolio', 'design', 'policy', 'accuracy_on_repayers', 'breakeven_accuracy_on_defaulters', 'derived_from_reported_gains']].round(3).to_string(index=False))
    print("\nBreak-even requires the reviewer to REJECT at least this share of defaulters (given the share of repayers approved correctly). Values above 1 mean break-even is impossible.")

In [ ]:
# ---------- R15: Spearman vs AUC (item 15), R16: empty sets (item 16) ----------
with section('R15 Spearman versus AUC'):
    from scipy.stats import spearmanr
    rows = []
    for dk, v in PDES.items():
        te = np.isin(yr, v['tests'])
        for fs in ('P0 with lender rate', 'P3 applicant-only rich'):
            rh = fit_reg(Xc[fs], roi, yr <= v['tr']); r0 = te & (y == 0); r1 = te & (y == 1)
            rows.append(dict(design=dk, features=fs, spearman_all=spearmanr(rh[te], roi[te]).correlation, spearman_repaid_only=spearmanr(rh[r0], roi[r0]).correlation, spearman_defaulted_only=spearmanr(rh[r1], roi[r1]).correlation,
                             AUC_profit_positive=roc_auc_score((pi[te] > 0).astype(int), rh[te]), AUC_repaid_vs_default=roc_auc_score((y[te] == 0).astype(int), rh[te]), share_profit_positive_among_defaults=float((pi[r1] > 0).mean()), mean_pred_return_repaid=rh[r0].mean(), mean_pred_return_default=rh[r1].mean()))
    if HAS_LCPAY:
        roiL = (LC['profit'] / LC['funded_amnt']).values; piL = LC['profit'].values; teL_ = yrL >= 2013
        for fs in ('F0', 'F3'):
            rh = fit_reg(XLc[fs], roiL, yrL <= 2011); r0 = teL_ & (yL == 0); r1 = teL_ & (yL == 1)
            rows.append(dict(design='Lending Club 2013-15', features=fs, spearman_all=spearmanr(rh[teL_], roiL[teL_]).correlation, spearman_repaid_only=spearmanr(rh[r0], roiL[r0]).correlation, spearman_defaulted_only=spearmanr(rh[r1], roiL[r1]).correlation,
                             AUC_profit_positive=roc_auc_score((piL[teL_] > 0).astype(int), rh[teL_]), AUC_repaid_vs_default=roc_auc_score((yL[teL_] == 0).astype(int), rh[teL_]), share_profit_positive_among_defaults=float((piL[r1] > 0).mean()), mean_pred_return_repaid=rh[r0].mean(), mean_pred_return_default=rh[r1].mean()))
    R15 = pd.DataFrame(rows); save(R15, 'R15_spearman_vs_auc'); print(R15.round(3).T.to_string())
with section('R16 empty sets'):
    rows = []
    def share(port, dk, p, yy, ca, te, label):
        S = conformal_sets(p, yy, ca)
        for meth, SS in S.items():
            s = SS[te]; rows.append(dict(portfolio=port, design=dk, method=meth, period=label, empty=float((~s.any(1)).mean()), only_0=float((s[:, 0] & ~s[:, 1]).mean()), only_1=float((~s[:, 0] & s[:, 1]).mean()), both=float(s.all(1).mean())))
    share('Lending Club', '-', P_L[BASE_L], yL, yrL == 2012, yrL >= 2013, '2013-15')
    for dk, v in PDES.items(): share('Prosper', dk, P_P[(dk,) + BASE_P], y, yr == v['ca'], np.isin(yr, v['tests']), 'test')
    R16 = pd.DataFrame(rows); save(R16, 'R16_empty_set_shares'); print(R16.round(4).to_string(index=False))

In [ ]:
# ---------- R17: in-text numbers recomputed from the data (item 17) ----------
with section('R17 number audit'):
    CHK = []
    def chk(item, claimed, got, tol):
        CHK.append(dict(item=item, manuscript=claimed, recomputed=round(float(got), 4), abs_diff=round(abs(claimed - got), 4), result='PASS' if abs(claimed - got) <= tol else 'CHECK'))
    man_lc = {2007: (251, .179), 2008: (1554, .156), 2009: (4702, .126), 2010: (8195, .096), 2011: (13541, .103), 2012: (41703, .134), 2013: (95371, .120), 2014: (152853, .133), 2015: (263943, .144)}
    for yy, (n_, dr_) in man_lc.items(): m = yrL == yy; chk(f'LC {yy} loans', n_, m.sum(), 0.5); chk(f'LC {yy} default rate', dr_, yL[m].mean(), 0.0006)
    gm_ = LC.groupby('year')['grade'].apply(lambda s: (s == 'A').mean()); chk('LC grade A share 2011', .39, gm_[2011], 0.006); chk('LC grade A share 2013', .17, gm_[2013], 0.006)
    chk('LC total loans', 582113, len(LC), 0.5); chk('LC default rate overall', .1348, yL.mean(), 0.0006)
    man_pr = {2006: (5902, .394), 2007: (11460, .395), 2008: (11551, .330), 2009: (2047, .152), 2010: (5599, .167)}
    for yy, (n_, dr_) in man_pr.items(): m = yr == yy; chk(f'Prosper {yy} loans', n_, m.sum(), 0.5); chk(f'Prosper {yy} default rate', dr_, y[m].mean(), 0.0006)
    m = (yr == 2011); chk('Prosper 2011Q1 loans', 1375, m.sum(), 0.5); chk('Prosper 2011Q1 default rate', .214, y[m].mean(), 0.0006); chk('Prosper loans total', 37934, len(f), 0.5); chk('Prosper default rate overall', .322, y.mean(), 0.0006)
    known = f['reveal'].values <= (2009 - 2006) * 12 + 12; cal = yr == 2009; chk('Prosper calibration loans known at deployment', 59, (cal & known).sum(), 0.5)
    print(f"Prosper 2009 calibration loans known at deployment: {int((cal & known).sum())}, of which defaults: {int(y[cal & known].sum())} ({100 * y[cal & known].mean():.1f}%)")
    CHKD = pd.DataFrame(CHK); save(CHKD, 'R17_number_audit'); print(CHKD[CHKD.result != 'PASS'].to_string(index=False) if (CHKD.result != 'PASS').any() else 'all recomputable numbers PASS'); print(CHKD.result.value_counts().to_dict())
    print("\nNot recomputable here (need payments): LC mean profit, returns on funded money, profit gains.")

In [ ]:
lines = [f"REVIEWER CHECKS SUMMARY ({time.strftime('%Y-%m-%d %H:%M')}), failed sections: {FAILED or 'none'}"]
(OUT/'reviewer_summary.txt').write_text("\n".join(lines)); print(f"FAILED SECTIONS: {FAILED or 'none'}"); print("Send me this file:", shutil.make_archive(str(Q2/'outputs_reviewer'), 'zip', OUT))